# Solutions · Chapter 01-04 · pandas I

Worked answers with reasoning. E9 and E14 are the two that generalise furthest - one traces a
single space character through a year of nightly runs, the other builds the sixty-second check as
a function you can keep.

Self-contained: run from the top with a fresh kernel.

In [ ]:
import io

import numpy as np
import pandas as pd

RAW_CSV = '''sensor,site,temp_c,rentals,reading_date
s1,north,"18,5",1200,2024-03-01
s2,north,"24,1",980,2024-03-01
s3,south,"31,7","1,050",2024-03-02
s4,south,"9,0",870,2024-03-02
s5,east,"27,3",1130,2024-03-03
s6,East ,"19,4",940,2024-03-03
'''

df = pd.read_csv(io.StringIO(RAW_CSV))
clean = df.assign(
    temp_c=lambda d: pd.to_numeric(d["temp_c"].str.replace(",", ".", regex=False), errors="coerce"),
    rentals=lambda d: pd.to_numeric(d["rentals"].str.replace(",", "", regex=False), errors="coerce"),
    reading_date=lambda d: pd.to_datetime(d["reading_date"]),
    site=lambda d: d["site"].str.strip().str.lower(),
)
print(clean.dtypes.to_string())

## E1 · Series versus DataFrame

A **Series** is one column: an array of values plus an index. A **DataFrame** is a table: several
Series sharing one index, each with a name.

`df["a"]` gives a Series - one dimension, no column name in the shape. `df[["a"]]` gives a
DataFrame with one column - two dimensions.

**Why it matters in practice:** scikit-learn wants `X` two-dimensional and `y` one-dimensional. So
`X = df[["temp_c"]]` and `y = df["rentals"]`, and the "expected 2D array, got 1D array instead"
error almost always means a pair of brackets is missing. It is the same `(n,)` versus `(n, 1)`
distinction from 01-03, wearing pandas clothing.

## E2 · The three things in `.info()`

1. **The row count.** Does it match what the file should contain? Catches a truncated read, a
   header row swallowed as data, a file that was still being written.
2. **The non-null counts per column.** Any column below the row count has missing values - you
   learn this before an aggregation quietly drops rows or returns `NaN`.
3. **The dtype of every column.** Catches the chapter's failure: numbers loaded as text, dates
   loaded as text, an ID column turned into a float (which loses leading zeros and can round
   large values).

The order matters. A wrong row count makes everything else moot, and missing values change how you
read the dtypes - a numeric column with one bad value becomes text, so "wrong dtype" and "missing
data" are usually the same finding seen from two sides.

## E3 · Why one raised and the other lied

`mean` has no definition for text, so pandas refuses. `max` does - text has an alphabetical order -
so pandas obliges, comparing character by character and returning `'9,0'`.

The difference is not that one operation is better implemented. It is that **one is undefined on
text and the other is defined but means something else.**

**Which behaviour would I rather have?** The `TypeError`, every time, and it is not close. An
error stops the notebook at the line that caused it, with a message naming the cause. A wrong
maximum flows into a report, a filter, a feature, a model - and is discovered weeks later by
someone who has to work backwards through five steps to find it, if they find it at all.

**The transferable principle:** *prefer loud failure to quiet plausibility*, and when a library
gives you the choice - `errors="raise"` versus `errors="coerce"`, strict versus permissive parsing
- pick loud during development. Silence is only appropriate once you have decided, deliberately,
what the silent case should do.

## E4 · Sorting text that looks like numbers

In [ ]:
values = ["1200", "980", "31,7", "9,0", "105"]
as_text = sorted(values)
as_numbers = sorted(values, key=lambda s: float(s.replace(",", ".")) if "," in s else float(s))

print("alphabetical:", as_text)
print("numeric     :", as_numbers)
print("positions that differ:", sum(a != b for a, b in zip(as_text, as_numbers)), "of 5")

**All five positions differ.**

Alphabetical order compares character by character: `"105"` beats `"1200"` because `'0'` comes
before `'2'` at the third character; `"9,0"` beats `"980"` because a comma sorts before `'8'`.
Nothing about the result resembles numeric order.

**Why this exercise is worth doing by hand:** it makes concrete what "silently wrong" means. A
top-ten table built from this column is not slightly off - it is unrelated to the question asked,
and it will look completely normal on a dashboard.

## E5 · Labels after filtering

In [ ]:
demo = pd.DataFrame({"x": [1, -1, -1, 5, 9]})
d = demo[demo["x"] > 0]
print("surviving index:", list(d.index))

print("d.loc[3]  ->", d.loc[3]["x"], "  (the row originally at label 3)")
print("d.iloc[1] ->", d.iloc[1]["x"], "  (the second row of d, which is label 3)")
for expr in ("d.iloc[3]", "d.loc[1]"):
    try:
        eval(expr)
    except Exception as exc:
        print(f"{expr:<10} -> {type(exc).__name__}")

- **`d.loc[3]`** works and gives the row that was originally at label 3. Labels survive filtering.
- **`d.iloc[3]`** raises `IndexError`. `d` has three rows, so the only valid positions are 0, 1 and
  2.
- **`d.loc[1]`** raises `KeyError`. Label 1 was filtered out and no longer exists.

Both errors are good news, because they are loud. The dangerous case is the one that does *not*
raise: on the original `demo`, `.loc[3]` and `.iloc[3]` return the same row, so code that confuses
them works perfectly until the day something upstream filters or sorts the frame. Then it silently
returns a different row.

**The habit:** decide which you mean and say it. Almost always you mean "the row about this
thing", which is `.loc`.

## E6 · Load, repair, assert

In [ ]:
def load_and_check(csv_text, expected_dtypes, comma_decimal=()):
    """Load a CSV, repair German decimal commas in the named columns, assert every dtype."""
    frame = pd.read_csv(io.StringIO(csv_text))
    for column in comma_decimal:
        frame[column] = pd.to_numeric(frame[column].str.replace(",", ".", regex=False), errors="coerce")
        missing = int(frame[column].isna().sum())
        if missing:
            print(f"  warning: {missing} value(s) in {column!r} could not be parsed")
    for column, expected in expected_dtypes.items():
        actual = str(frame[column].dtype)
        assert actual == expected, f"{column!r} is {actual}, expected {expected}"
    return frame


ok = load_and_check(RAW_CSV, {"temp_c": "float64", "sensor": "str"}, comma_decimal=["temp_c"])
print("loaded:", ok.shape, "temp_c dtype", ok["temp_c"].dtype)

try:
    load_and_check(RAW_CSV, {"rentals": "int64"}, comma_decimal=["temp_c"])
except AssertionError as exc:
    print("caught:", exc)

The assert catches `rentals` immediately: it is still text because of the `"1,050"` thousands
separator, and the function refuses to hand back a frame that does not match the promise.

Three deliberate choices in that function, each worth copying:

- **The repair is explicit per column.** No global `decimal=","`, because the chapter showed that
  turning it on globally silently divides a thousands-separated value by a thousand.
- **Unparseable values are counted and reported.** `errors="coerce"` without that count is how a
  column quietly becomes half missing.
- **The assert names the column and both dtypes.** An assert whose message is just `AssertionError`
  costs you the debugging time it was supposed to save.

**What this is a small version of:** a data contract - a machine-checked statement of what a file
must look like before anything is allowed to use it. Chapter 13-05 builds the real one.

## E7 · Filtering on several conditions

In [ ]:
result = (clean.loc[clean["reading_date"].isin(pd.to_datetime(["2024-03-02", "2024-03-03"]))
                    & clean["site"].str.startswith("s"),
                    ["sensor", "site", "temp_c"]]
          .sort_values("temp_c", ascending=False))
print(result)

Both south readings, hottest first. `east` is excluded by the name test and the 1 March rows by
the date test.

Two details worth noting:

- **The dates had to be converted** before comparison. `clean["reading_date"]` holds timestamps, and
  comparing a timestamp column against the *strings* `"2024-03-02"` works in some places and not
  others. Being explicit with `pd.to_datetime` removes the ambiguity. An alternative that reads
  better for a range: `clean["reading_date"].between("2024-03-02", "2024-03-03")`.
- **`.str.startswith` gives a boolean mask**, so it combines with `&` like any other condition.
  The whole `.str` family - `contains`, `strip`, `lower`, `replace`, `len` - works this way, and it
  is how text columns are filtered and cleaned throughout module 02.

## E8 · Is the 9.0 real?

Three questions, in the order I would ask them:

1. **What are the units, and is 9.0 possible?** Nine degrees in March in a European city is
   entirely plausible - so this one is probably fine. The same value in a column of body
   temperatures would be impossible, and impossible values are the easiest defect to find. Always
   check the minimum and maximum against what the world allows.
2. **Is it a sentinel?** Placeholder values for "not recorded" are often round: 0, -1, -999, 99,
   9. If several rows shared exactly 9.0, or if the column also contained a suspicious 0, I would
   suspect a code rather than a measurement. Sentinels load as valid numbers and drag every
   average toward them.
3. **What else is unusual about that row?** Sensor s4, at the same site and on the same day as the
   31.7 reading. Two sensors twenty-two degrees apart, at one site, on one day, is a stronger
   signal than either number alone - a faulty sensor, a unit mix-up, or a sensor in the shade
   versus the sun.

**The judgement, which is what 02-05 is about:** an outlier is not a category of value, it is a
question. Some are errors to remove, some are the most important rows in the dataset, and the
only way to tell is to find out how the number was produced.

## E9 · One space, one year, one broken dashboard

**The chain of events:**

1. Every previous night, every value in `rentals` parsed as an integer, so `read_csv` inferred
   `int64` and everything downstream worked.
2. Last night one field read `"1 050"`. That is not a number in any convention pandas tries, so the
   **whole column** fell back to text. One bad cell changes the type of the entire column - dtypes
   belong to columns, not to values.
3. Nothing raised. `read_csv` succeeded, the row count was right, the pipeline continued.
4. The monthly total is computed with `.sum()`, which on text **concatenates**. Thirty days of
   rental counts became one long string, and the dashboard displayed it, because a string is a
   perfectly displayable thing.

**The single line of defence:** an assert on the dtype immediately after loading.

```python
assert df["rentals"].dtype == "int64", f"rentals arrived as {df['rentals'].dtype}"
```

**Where it goes: in the load step, before anything else touches the frame.** That is the point at
which the file becomes data, and it is the only place where the failure is one row away from its
cause. A check at the dashboard would tell you something is wrong; a check at the load tells you
what.

**The deeper lesson.** A year of successful runs is not evidence that the code is correct - only
that the inputs have been well behaved. The pipeline had no opinion about what a valid file looked
like, so it accepted an invalid one. Chapter 13-05 calls the fix a data contract, and this is the
one-line version of it.

## E10 · "Isn't `.info()` a waste of time on a familiar dataset?"

> The check is not about the dataset, it is about the file - and the file is new every time. A
> familiar dataset acquires a new column, loses one, changes an encoding, or has one malformed cell
> that flips a whole column to text, and none of those raise an error at load. It costs one line
> and a second of reading, against the cost of building a model on a column that is silently text
> or a date that is silently a string. I would rather spend the second every time than spend an
> afternoon working backwards from an absurd number on a dashboard.

**What is really being tested:** whether you distinguish *the data* from *this delivery of the
data*. Everything that goes wrong in production goes wrong at that boundary.

## E11 · Three checks on the morning file

1. **Schema.** Exactly the expected columns, with the expected names and dtypes after the known
   repairs. Extra or missing columns and wrong types are the cheapest failures to detect and the
   most damaging to miss.
2. **Volume and completeness.** Row count within a plausible range of the recent daily average -
   an empty or half-written file is common - and the count of missing values per column no higher
   than usual. A column that suddenly becomes 40% missing is a broken upstream field, not noise.
3. **Range and validity.** Each numeric column inside a range the world allows (temperature between
   -30 and 50, rentals non-negative), dates within the expected day, and categorical columns
   containing only known values - which would have caught `East ` on day one.

**What should happen when one fails: stop, and alert a person.** Do not repair silently, and do not
carry on with the previous day's file, because both produce a dashboard that looks fine and is
wrong. Quarantine the file, keep the last known-good output, and say clearly which check failed and
on which rows.

**The one people forget:** the checks themselves need to be visible when they *pass*. A quality
check that only speaks up on failure is indistinguishable from a quality check that has been
silently broken for three months.

## E12 · The hospital export

| Column | Dtype you want | How to get there | What could go wrong silently |
|---|---|---|---|
| `age` | `float64` (or a nullable integer) | `pd.to_numeric(..., errors="coerce")`, turning `"unknown"` into `NaN` | The count of coerced values is never checked. If 30% of ages are `"unknown"`, dropping those rows quietly removes a group - and it is very unlikely to be a random group |
| `weight_kg` | `float64` | Replace the comma, then `to_numeric` | A value like `"1,234"` meaning 1234 grams, or a stray unit suffix - the comma repair turns it into 1.234, which is a plausible weight for nothing |
| `admitted` | `bool` | `.map({"1": True, "0": False})` | `.astype(bool)` on the *strings* makes everything `True`, because any non-empty string is truthy. This raises nothing and inverts your entire analysis for one class |

**The `admitted` row is the important one**, because `astype(bool)` is the obvious move and it is
catastrophically wrong. An explicit `map` with a dictionary fails loudly on any value you did not
anticipate - `"Y"`, `"true"`, `" 1"` - which is exactly what you want.

**And a fourth thing that is not a dtype at all:** `age` and `weight_kg` need a plausibility check
after conversion. An age of 300 or a weight of 7 kg parses perfectly and means the record is
broken, or means a neonate - and telling those apart needs someone who knows the domain, not a
type check. 02-04.

## E13 · Explaining it to the supplier

> Your export writes numbers the German way - 18,5 with a comma - and mixes that with 1,050 where
> the comma means a thousand. Our tools read both as text, so totals and maximums came out wrong
> without any error. Could you export decimals with a point and no thousands separator, and dates
> as 2024-03-01? Same file, same columns, just those three conventions.

(63 words.)

**Why the ask is specific:** "your CSV is broken" starts an argument, because from their side it
opens correctly in Excel. Naming the three conventions and giving an example of each turns it into
a five-minute settings change.

**And the general point:** this is the cheapest fix available. Repairing the file in your notebook
is a permanent maintenance cost that every future colleague inherits; fixing the export is done
once. Always ask where a problem is cheapest to fix, and it is usually upstream of you.

## E14 · A profile function you can keep

In [ ]:
def profile(frame):
    """One row per column: dtype, missing, distinct, and whether tidying text would merge values."""
    rows = []
    for name in frame.columns:
        column = frame[name]
        is_text = pd.api.types.is_string_dtype(column) or column.dtype == object
        tidied = column.str.strip().str.lower().nunique() if is_text else np.nan
        rows.append({"column": name, "dtype": str(column.dtype),
                     "missing": int(column.isna().sum()), "distinct": int(column.nunique()),
                     "distinct_if_tidied": tidied})
    return pd.DataFrame(rows).set_index("column")


profile(df)

Run on the **raw** frame, this finds all three problems in one call:

1. **`temp_c` and `rentals` have dtype `str`** when they should be numbers - the failure lab's
   entire bug, visible in a column of the output.
2. **`reading_date` is `str`**, so no date arithmetic, no resampling, and any sort works only by
   luck of the ISO format.
3. **`site` has 4 distinct values but only 3 after tidying** - `East ` and `east` are the same
   place, recorded twice. This is the one that is genuinely hard to see by eye, because the
   difference is a trailing space, and it is the one that silently splits a group in two during a
   `groupby`.

**Why write it as a function rather than doing it by eye.** Six rows fit on a screen. Sixty
thousand do not, and a hundred columns certainly do not. A profile scales, it is the same every
time, and it can be diffed between yesterday's file and today's - which turns "look at the data"
from a discipline you have to remember into a line you always run.

**What it does not do**, and what module 02 adds: distributions, relationships between columns,
plausibility of ranges, and whether the missing values are missing for a reason. A profile finds
defects in the *recording*. Finding defects in the *meaning* takes a person who asks where the
data came from - which is 02-02.

---

## Where to go next

Back to the chapter for the mastery check and flashcards, then **01-05 · pandas II: grouping,
joining, timestamps**.